# MathVision Preprocessing Notebook

Spec A implementation for pairing quality labeling using tutor feedback text.


## 1) Imports and setup


In [1]:
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2) Preprocessing parameters


In [2]:
PREPROCESSING_PATHS = {
    "students": "../data/raw/students.csv",
    "tutors": "../data/raw/tutors.csv",
    "pairings_raw": "../data/raw/pairings_raw.csv",
    "pairings_labeled": "../data/pre-processed/pairings_labeled.csv",
}

POSITIVE_PHRASES = [
    "improving understanding",
    "good progress",
    "grasped the concept",
    "more confident",
    "able to solve independently",
    "ready for next topic",
]

NEGATIVE_PHRASES = [
    "still struggling",
    "weak understanding",
    "confused about topic",
    "many mistakes",
    "needs repeated explanation",
    "unable to solve independently",
]

REQUIRED_RAW_COLUMNS = [
    "pairing_id",
    "student_id",
    "tutor_id",
    "session_date",
    "duration_hours",
    "tutor_feedback_text",
]

PAIRINGS_LABELED_COLUMNS = [
    "pairing_id",
    "student_id",
    "tutor_id",
    "lessons_count",
    "total_hours",
    "avg_feedback_score",
    "positive_note_count",
    "negative_note_count",
    "good_pairing_label",
]


## 3) Preprocessing functions


In [3]:
def clean_feedback_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def validate_pairings_raw(df_raw):
    missing_cols = [c for c in REQUIRED_RAW_COLUMNS if c not in df_raw.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    for col in ["pairing_id", "student_id", "tutor_id", "tutor_feedback_text"]:
        if df_raw[col].isna().any():
            bad_idx = int(df_raw[df_raw[col].isna()].index[0])
            raise ValueError(f"Column {col} has null at row index {bad_idx}")

    duration_numeric = pd.to_numeric(df_raw["duration_hours"], errors="coerce")
    if duration_numeric.isna().any():
        bad_idx = int(duration_numeric[duration_numeric.isna()].index[0])
        bad_val = df_raw.loc[bad_idx, "duration_hours"]
        raise ValueError(f"duration_hours is non-numeric at row index {bad_idx}: {bad_val}")
    if (duration_numeric < 0).any():
        bad_idx = int(duration_numeric[duration_numeric < 0].index[0])
        bad_val = float(duration_numeric.loc[bad_idx])
        raise ValueError(f"duration_hours is negative at row index {bad_idx}: {bad_val}")


def build_reference_vectors(positive_phrases, negative_phrases, vectorizer=None):
    if vectorizer is None:
        vectorizer = TfidfVectorizer()

    cleaned_positive = [clean_feedback_text(p) for p in positive_phrases]
    cleaned_negative = [clean_feedback_text(p) for p in negative_phrases]
    vectorizer.fit(cleaned_positive + cleaned_negative)

    positive_reference_vector = vectorizer.transform([" ".join(cleaned_positive)])
    negative_reference_vector = vectorizer.transform([" ".join(cleaned_negative)])
    return vectorizer, positive_reference_vector, negative_reference_vector


def score_session_feedback(tutor_feedback_text, vectorizer, positive_reference_vector, negative_reference_vector):
    cleaned_note = clean_feedback_text(tutor_feedback_text)
    note_vector = vectorizer.transform([cleaned_note])

    similarity_positive = float(cosine_similarity(note_vector, positive_reference_vector)[0][0])
    similarity_negative = float(cosine_similarity(note_vector, negative_reference_vector)[0][0])
    feedback_score = similarity_positive - similarity_negative

    return similarity_positive, similarity_negative, feedback_score


def label_pairing(avg_feedback_score):
    return 1 if avg_feedback_score > 0 else 0


def aggregate_pairings(scored_sessions_df):
    grouped = (
        scored_sessions_df
        .groupby(["pairing_id", "student_id", "tutor_id"], as_index=False)
        .agg(
            lessons_count=("pairing_id", "size"),
            total_hours=("duration_hours", "sum"),
            avg_feedback_score=("feedback_score", "mean"),
            positive_note_count=("feedback_score", lambda s: int((s > 0).sum())),
            negative_note_count=("feedback_score", lambda s: int((s < 0).sum())),
        )
    )
    grouped["good_pairing_label"] = grouped["avg_feedback_score"].apply(label_pairing)
    return grouped[PAIRINGS_LABELED_COLUMNS]


def preprocess_pairings_raw(df_raw, positive_phrases=None, negative_phrases=None):
    if positive_phrases is None:
        positive_phrases = POSITIVE_PHRASES
    if negative_phrases is None:
        negative_phrases = NEGATIVE_PHRASES

    validate_pairings_raw(df_raw)
    df = df_raw.copy()
    df["duration_hours"] = pd.to_numeric(df["duration_hours"], errors="raise")
    df["cleaned_feedback_text"] = df["tutor_feedback_text"].apply(clean_feedback_text)

    vectorizer, positive_ref_vec, negative_ref_vec = build_reference_vectors(
        positive_phrases=positive_phrases,
        negative_phrases=negative_phrases,
    )

    scored_rows = df["cleaned_feedback_text"].apply(
        lambda txt: score_session_feedback(txt, vectorizer, positive_ref_vec, negative_ref_vec)
    )
    scores_df = pd.DataFrame(
        scored_rows.tolist(),
        columns=["similarity_positive", "similarity_negative", "feedback_score"],
        index=df.index,
    )
    df = pd.concat([df, scores_df], axis=1)

    labeled_df = aggregate_pairings(df)
    return labeled_df, df


def run_preprocessing_from_csv(
    pairings_raw_path,
    output_path,
    positive_phrases=None,
    negative_phrases=None,
):
    df_raw = pd.read_csv(pairings_raw_path)
    labeled_df, scored_sessions_df = preprocess_pairings_raw(
        df_raw=df_raw,
        positive_phrases=positive_phrases,
        negative_phrases=negative_phrases,
    )
    labeled_df.to_csv(output_path, index=False)
    return labeled_df, scored_sessions_df


## 4) Validation and unit-like checks


In [4]:
# Unit-like checks for cleaning and score direction
assert clean_feedback_text("Student showed Improvement in Fractions!") == "student showed improvement in fractions"

vectorizer, pos_ref, neg_ref = build_reference_vectors(POSITIVE_PHRASES, NEGATIVE_PHRASES)
_, _, pos_score = score_session_feedback(
    "Student made good progress and is more confident.",
    vectorizer,
    pos_ref,
    neg_ref,
)
_, _, neg_score = score_session_feedback(
    "Student is still struggling and unable to solve independently.",
    vectorizer,
    pos_ref,
    neg_ref,
)
assert pos_score > 0
assert neg_score < 0
print("Cleaning and score direction checks passed.")


Cleaning and score direction checks passed.


In [5]:
# Aggregation and label checks with toy sessions
toy_df = pd.DataFrame(
    [
        {
            "pairing_id": "P001",
            "student_id": "S001",
            "tutor_id": "T001",
            "session_date": "2026-03-01",
            "duration_hours": 1.5,
            "tutor_feedback_text": "Good progress and improving understanding.",
        },
        {
            "pairing_id": "P001",
            "student_id": "S001",
            "tutor_id": "T001",
            "session_date": "2026-03-08",
            "duration_hours": 1.0,
            "tutor_feedback_text": "Still struggling with many mistakes.",
        },
        {
            "pairing_id": "P002",
            "student_id": "S002",
            "tutor_id": "T002",
            "session_date": "2026-03-02",
            "duration_hours": 2.0,
            "tutor_feedback_text": "Grasped the concept and ready for next topic.",
        },
    ]
)

toy_labeled, toy_scored = preprocess_pairings_raw(toy_df)

p001 = toy_labeled[toy_labeled["pairing_id"] == "P001"].iloc[0]
assert int(p001["lessons_count"]) == 2
assert abs(float(p001["total_hours"]) - 2.5) < 1e-9
assert int(p001["positive_note_count"] + p001["negative_note_count"]) <= int(p001["lessons_count"])
assert int(p001["good_pairing_label"]) == (1 if float(p001["avg_feedback_score"]) > 0 else 0)

assert list(toy_labeled.columns) == PAIRINGS_LABELED_COLUMNS
print("Aggregation and labeling checks passed.")


Aggregation and labeling checks passed.


In [6]:
# Strict validation checks (expected failures)
missing_col_df = pd.DataFrame(
    [{
        "pairing_id": "P003",
        "student_id": "S003",
        "tutor_id": "T003",
        "session_date": "2026-03-03",
        "duration_hours": 1.0,
    }]
)
try:
    validate_pairings_raw(missing_col_df)
    raise AssertionError("Expected missing-column validation failure")
except ValueError as exc:
    assert "Missing required columns" in str(exc)

null_feedback_df = pd.DataFrame(
    [{
        "pairing_id": "P004",
        "student_id": "S004",
        "tutor_id": "T004",
        "session_date": "2026-03-04",
        "duration_hours": 1.0,
        "tutor_feedback_text": None,
    }]
)
try:
    validate_pairings_raw(null_feedback_df)
    raise AssertionError("Expected null-feedback validation failure")
except ValueError as exc:
    assert "tutor_feedback_text" in str(exc)

invalid_duration_df = pd.DataFrame(
    [{
        "pairing_id": "P005",
        "student_id": "S005",
        "tutor_id": "T005",
        "session_date": "2026-03-05",
        "duration_hours": "bad",
        "tutor_feedback_text": "Good progress.",
    }]
)
try:
    validate_pairings_raw(invalid_duration_df)
    raise AssertionError("Expected duration validation failure")
except ValueError as exc:
    assert "duration_hours" in str(exc)

print("Strict validation checks passed.")


Strict validation checks passed.


## 5) Run preprocessing on raw data


In [7]:
# Optional run block: executes once pairings_raw.csv is provided
raw_path = Path(PREPROCESSING_PATHS["pairings_raw"])
if raw_path.exists():
    labeled_df, scored_df = run_preprocessing_from_csv(
        pairings_raw_path=PREPROCESSING_PATHS["pairings_raw"],
        output_path=PREPROCESSING_PATHS["pairings_labeled"],
    )
    print(f"Saved {len(labeled_df)} rows to {PREPROCESSING_PATHS['pairings_labeled']}")
    print(labeled_df.head())
else:
    print(
        f"Skipped full preprocessing run: {PREPROCESSING_PATHS['pairings_raw']} not found. "
        "Provide the raw file, then rerun this cell."
    )


Saved 400 rows to ../data/pre-processed/pairings_labeled.csv
  pairing_id student_id tutor_id  lessons_count  total_hours  \
0      P0001       S031     T004              1          1.5   
1      P0002       S050     T011              1          2.0   
2      P0003       S034     T004              1          1.0   
3      P0004       S010     T016              1          1.5   
4      P0005       S015     T016              1          1.5   

   avg_feedback_score  positive_note_count  negative_note_count  \
0           -0.155528                    0                    1   
1            0.357054                    1                    0   
2            0.357054                    1                    0   
3           -0.142565                    0                    1   
4            0.357054                    1                    0   

   good_pairing_label  
0                   0  
1                   1  
2                   1  
3                   0  
4                   1  
